# train_hopper_sac.ipynb
**SAC Robustness Check — Hopper-v4**  
Methodology: v12.2 Addendum | Platform: Kaggle (CPU)  
Conditions: M1, M2, M3 | Seeds: 5 | Budget: 1,000,000 steps/seed/condition  

SAC uses n_envs=1 (off-policy, replay buffer) — no VecEnv wrapper required.  
Network: [256, 256] ReLU + Adam, consistent with PPO branch.  
Purpose: cross-algorithm robustness check; not a benchmark.

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────────
!pip install stable-baselines3[extra] gymnasium[mujoco] --quiet

In [ ]:
import os
import time
import numpy as np
import torch
import gymnasium as gym

from stable_baselines3 import SAC

torch.set_num_threads(os.cpu_count())

# ── Config ─────────────────────────────────────────────────────────────────────
ENV_ID      = "Hopper-v4"
OUTPUT_DIR  = "/kaggle/working/results/mujoco/hopper"
TOTAL_STEPS = 1_000_000
SEEDS       = [0, 1, 2, 3, 4]

CONDITIONS = {
    "M1": {"timelimit": 1000, "reset_noise_scale": 0.005},
    "M2": {"timelimit": 250,  "reset_noise_scale": 0.005},
    "M3": {"timelimit": 1000, "reset_noise_scale": 0.02},
}

# Network locked to [256,256] ReLU Adam — consistent with PPO branch (v12.2 §3.4.5)
SAC_KWARGS = dict(
    learning_rate    = 3e-4,
    buffer_size      = 1_000_000,
    learning_starts  = 10_000,
    batch_size       = 256,
    tau              = 0.005,
    gamma            = 0.99,
    train_freq       = 1,
    gradient_steps   = 1,
    ent_coef         = "auto",
    target_entropy   = "auto",
    use_sde          = False,
    policy_kwargs    = dict(
        net_arch        = [256, 256],
        activation_fn   = torch.nn.ReLU,
        optimizer_class = torch.optim.Adam,
    ),
    verbose          = 1,
    device           = "cpu",
)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output: {OUTPUT_DIR}")
print(f"Conditions: {list(CONDITIONS.keys())} | Seeds: {SEEDS}")

In [ ]:
# ── Environment factory ────────────────────────────────────────────────────────
from gymnasium.wrappers import TimeLimit

def make_env(timelimit, reset_noise_scale, seed):
    """Single env, no VecEnv — SAC is off-policy."""
    env = gym.make(
        ENV_ID,
        reset_noise_scale=reset_noise_scale,
    )
    env = TimeLimit(env, max_episode_steps=timelimit)
    env.reset(seed=seed)
    return env

In [ ]:
# ── Training loop ──────────────────────────────────────────────────────────────
print("=" * 60)
print(f"SAC TRAINING — {ENV_ID}")
print(f"Steps/run: {TOTAL_STEPS:,} | n_envs: 1 (SAC off-policy) | Device: cpu")
print("=" * 60)

total_start = time.time()

for cond_name, cond_cfg in CONDITIONS.items():
    print(f"\n── Condition {cond_name} | TimeLimit={cond_cfg['timelimit']} | "
          f"noise={cond_cfg['reset_noise_scale']} ──")

    for seed in SEEDS:
        save_path = os.path.join(OUTPUT_DIR, f"{cond_name}_sac_seed{seed}.zip")

        if os.path.exists(save_path):
            print(f"  [skip] {cond_name} seed={seed} — already trained")
            continue

        run_start = time.time()
        print(f"  [train] {cond_name} seed={seed} ...", flush=True)

        env   = make_env(cond_cfg["timelimit"], cond_cfg["reset_noise_scale"], seed)
        model = SAC("MlpPolicy", env, seed=seed, **SAC_KWARGS)
        model.learn(total_timesteps=TOTAL_STEPS)
        model.save(save_path)
        env.close()

        elapsed = time.time() - run_start
        print(f"  [done]  {cond_name} seed={seed} → {save_path} ({elapsed/60:.1f} min)")

total_elapsed = time.time() - total_start
print(f"\n{'='*60}")
print(f"All training complete in {total_elapsed/60:.1f} min ({total_elapsed/3600:.2f} h)")
print(f"{'='*60}")

In [ ]:
# ── Inventory ──────────────────────────────────────────────────────────────────
print("Output inventory:")
for cond_name in CONDITIONS:
    for seed in SEEDS:
        path   = os.path.join(OUTPUT_DIR, f"{cond_name}_sac_seed{seed}.zip")
        status = "✓" if os.path.exists(path) else "✗ MISSING"
        print(f"  {status}  {cond_name}_sac_seed{seed}.zip")